**Esercizio 1**: quanto costa `dbscan`? 
facendo riferimento al codice sotto, faccio $n$ iterazioni, con $n$ il numero di punti.

l'algoritmo vuole classificare i punti nel seguente modo:
* enumero ed estraggo i punti uno alla volta
* se non e' stato etichettato ed ho abbastanza vicini allora quello diventa un cluster
* se e' un cluster allora devo etichettare i vicini con quel nome

l'algoritmo guarda lo stesso punto al piu' due volte, quando:
* e' stato etichettato come `Noise`
* viene identificato come raggiungibile da qualche core


Dunque ho $n$ iterazioni. Ad ogni iterazione estraggo un punto `p`:
* se `p` non era etichettato, estraggo i punti a distanza `eps` da `p` in tempo $log N$ (`tree.query_ball_point`)
* al caso peggiore ogni nodo e' un cluster formato da solo se stesso

Ogni volta che identifico un cluster lo devo espandere, ipottiziamo che il numero di nodi vicini $N$ sia una costante $c = |N|$ che posso ignorare:
* devo guardare i vicini del nodo per vedere se e' un core e costa sempre `log N`

Dunque il costo e': $O(N * (log N + c log N)) = O(N log N)$


**Esercizio 2**. Si utilizzi il metodo descritto per trovare il valore di `eps` consigliato per i tre dataset dell'esempio precedente.

In [ ]:
import numpy as np
from scipy.spatial import KDTree

def dbscan(points, eps, min_pts):
    points = np.array(points)

    n = points.shape[0]
    tree = KDTree(points)

    c = -1  # c+1 sarà il 'nome' del prossimo cluster
    labels = [None]*n

    for i, p in enumerate(points):
        if labels[i] != None:
            continue
        N = tree.query_ball_point(p, eps) # indici in points dei punti a distanza eps da p
        if len(N) < min_pts:
            labels[i] = 'Noise'
            continue

        c += 1

        labels[i] = str(c)

        # estensione
        S = set(N) - set([i]) # nel caso medio |N| è considerato costante
        while len(S) > 0:
            j = S.pop()
            if labels[j] == 'Noise':
                labels[j] = str(c)
            if labels[j] != None:
                continue
            labels[j] = str(c)
            N = tree.query_ball_point(points[j], eps)
            if len(N) < min_pts:
                continue
            S = S.union(set(N) - set([j])) # costo lineare in max(S, |N|)
            # S.update( set(N) - set([j]) )  # meglio usare questa: costo lineare in |N|, quindi costante nel caso medio

    return labels